# Занятие 3, демо 2. Одна рекуррентность - и структура в матрице

Рекуррентность занятия: состояние стареет в $a_t$ раз и получает новую запись,
$S_t=a_tS_{t-1}+v_tk_t^\top$, читаем $y_t=S_tq_t$.

Тот же ответ можно получить матрицей коэффициентов, $Y=\Omega V$, где

$$
\Omega_{ti}=\mathbf 1[i\leq t]\left(\prod_{j=i+1}^{t}a_j\right)q_t^\top k_i.
$$

Вопрос: что именно рекуррентность оставляет в этой матрице.

In [ ]:
import torch

torch.set_num_threads(1)

"""Попарная матрица скалярной рекуррентности и её структура.

Рекуррентность занятия: S_t = a_t S_{t-1} + v_t k_t^T, чтение y_t = S_t q_t.
Здесь та же величина считается двумя способами - обходом по позициям и одной
матрицей коэффициентов - и видно, какую структуру рекуррентность оставляет в
этой матрице.
"""
import torch


def inputs(T, d_k=1, d_v=1, a_low=0.8, a_high=1.0, seed=0):
    """Входы и коэффициенты перехода a_t в отрезке [a_low, a_high]."""
    g = torch.Generator().manual_seed(seed)
    mk = lambda *s: torch.randn(*s, generator=g, dtype=torch.float64)
    a = a_low + (a_high - a_low) * torch.rand(T, generator=g, dtype=torch.float64)
    return mk(T, d_k), mk(T, d_k), mk(T, d_v), a


def recurrent(q, k, v, a):
    """Обход по позициям: состояние стареет в a_t раз и получает новую запись."""
    state = torch.zeros(v.shape[-1], k.shape[-1], dtype=q.dtype)
    out = []
    for t in range(q.shape[0]):
        state = a[t] * state + torch.outer(v[t], k[t])
        out.append(state @ q[t])
    return torch.stack(out)


def omega(q, k, a):
    """Коэффициенты Ω[t, i]: сколько записи i дошло до t и насколько она подходит.

    Произведение коэффициентов от i + 1 до t считается как отношение
    накопленных произведений, поэтому вся матрица получается без циклов.
    """
    T = q.shape[0]
    cumulative = torch.cumprod(a, dim=0)
    survival = cumulative.unsqueeze(1) / cumulative.unsqueeze(0)
    content = q @ k.T
    causal = torch.ones(T, T, dtype=q.dtype).tril()
    return causal * survival * content


def by_matrix(q, k, v, a):
    """Тот же выход, посчитанный одной матрицей: Y = Ω V."""
    return omega(q, k, a) @ v


def cross_block(matrix, m):
    """Блок через границу: строки после позиции m, столбцы до неё включительно.

    Позиции нумеруются с единицы, как в лекции: m = 2 означает границу после
    второй позиции.
    """
    return matrix[m:, :m]


def rank(matrix, tol=1e-9):
    """Ранг по сингулярным числам с явным порогом относительно наибольшего."""
    values = torch.linalg.svdvals(matrix.to(torch.float64))
    if values.numel() == 0 or values[0] == 0:
        return 0
    return int((values > tol * values[0]).sum())


def worst_cross_rank(matrix):
    """Наибольший ранг блока через границу по всем границам."""
    T = matrix.shape[0]
    return max(rank(cross_block(matrix, m)) for m in range(1, T))


def dense_causal(T, seed=0):
    """Произвольная нижнетреугольная матрица - без всякой рекуррентности."""
    g = torch.Generator().manual_seed(seed)
    return torch.randn(T, T, generator=g, dtype=torch.float64).tril()


def boundary_views(q, k, a, m):
    """Запросы справа и ключи слева с приклеенными коэффициентами перехода.

    Ровно то, что на слайде: множители перехода делятся в точке границы и
    уходят один в запрос, другой в ключ.
    """
    cumulative = torch.cumprod(a, dim=0)
    q_right = q[m:] * (cumulative[m:] / cumulative[m - 1]).unsqueeze(1)
    k_left = k[:m] * (cumulative[m - 1] / cumulative[:m]).unsqueeze(1)
    return q_right, k_left


def show(matrix, digits=4):
    """Печать матрицы построчно, без экспоненты."""
    for row in matrix.tolist():
        print("   ", "  ".join(f"{x:>{digits + 4}.{digits}f}" for x in row))

## Два способа посчитать одно и то же

Три позиции, одномерные ключи и значения, все запросы и ключи равны единице.
Записи: 2, 4 и 8. Коэффициенты перехода: половина и четверть.

In [ ]:
q = torch.ones(3, 1, dtype=torch.float64)
k = torch.ones(3, 1, dtype=torch.float64)
v = torch.tensor([[2.0], [4.0], [8.0]], dtype=torch.float64)
a = torch.tensor([1.0, 1 / 2, 1 / 4], dtype=torch.float64)

print("обходом по позициям:", recurrent(q, k, v, a).flatten().tolist())
print("одной матрицей:     ", by_matrix(q, k, v, a).flatten().tolist())
print()
print("матрица коэффициентов:")
show(omega(q, k, a))

## Коэффициенты связаны между собой

Четыре позиции, коэффициенты перехода 1/2, 1/3 и 1/4. Разрежем
последовательность после второй позиции и посмотрим на блок, который связывает
запросы справа от границы с ключами слева.

In [ ]:
a4 = torch.tensor([1.0, 1 / 2, 1 / 3, 1 / 4], dtype=torch.float64)
q4 = torch.ones(4, 1, dtype=torch.float64)
k4 = torch.ones(4, 1, dtype=torch.float64)

full = omega(q4, k4, a4)
print("вся матрица:")
show(full)

block = cross_block(full, 2)
print("\nблок через границу после позиции 2:")
show(block)
print("\nвторая строка, делённая на первую:",
      [round(x, 4) for x in (block[1] / block[0]).tolist()])
print("ранг блока:", rank(block))

## Состояние шире: ранг упирается в его ширину

Теперь $d_k=4$, двадцать четыре позиции, коэффициенты перехода случайные в
отрезке от 0,8 до 1. Сама матрица полного ранга. А блок через границу - нет.

In [ ]:
q, k, v, a = inputs(T=24, d_k=4, d_v=2, a_low=0.8, a_high=1.0, seed=0)
big = omega(q, k, a)

print("формы совпали с точностью:",
      float((recurrent(q, k, v, a) - by_matrix(q, k, v, a)).abs().max()))
print()
print("ранг всей матрицы:            ", rank(big), "из", big.shape[0])
print("ранг блока через середину:    ", rank(cross_block(big, 12)),
      "при размере блока", tuple(cross_block(big, 12).shape))
print("худший ранг по всем границам: ", worst_cross_rank(big))

## Это следствие рекуррентности, а не треугольности

Возьмём произвольную нижнетреугольную матрицу того же размера. Причинность у
неё та же, рекуррентности за ней нет.

In [ ]:
arbitrary = dense_causal(24)
print("произвольная причинная матрица")
print("  ранг блока через середину:    ", rank(cross_block(arbitrary, 12)))
print("  худший ранг по всем границам: ", worst_cross_rank(arbitrary))

## Откуда берётся ограничение

Множители перехода делятся в точке границы: часть уходит в запрос справа, часть
- в ключ слева. После этого блок через границу - произведение двух матриц
шириной $d_k$, и ранг больше $d_k$ быть не может.

In [ ]:
q_right, k_left = boundary_views(q, k, a, 12)
print("запросы справа:", tuple(q_right.shape), " ключи слева:", tuple(k_left.shape))
print("расхождение с блоком:",
      float((cross_block(big, 12) - q_right @ k_left.T).abs().max()))

## Сколько записи доживает до нужного места

Тот же множитель выживания отвечает на вопрос занятия 2: через сколько позиций
запись перестаёт быть слышна.

In [ ]:
print(f"  {'a':>6}  {'через 10':>10}  {'через 100':>11}  {'через 1000':>12}")
for value in (0.9, 0.99, 1.0):
    row = [value ** distance for distance in (10, 100, 1000)]
    print(f"  {value:>6}  {row[0]:>10.4g}  {row[1]:>11.4g}  {row[2]:>12.4g}")

## Что из этого следует

Рекуррентность с конечным состоянием и структурированная попарная матрица - не
две разные идеи, а два взгляда на одно вычисление. Ширина состояния ограничивает
ранг всего, что проходит через границу времени: сколько бы позиций ни было слева
и справа, разговор между ними идёт через узкое место.

Обратного не следует. Из того, что у матрицы есть структура, не следует ни
softmax, ни то, что две архитектуры с похожей структурой - одно и то же.